# SNC — Thesis Figures for One Model Version

Generates **every figure and table** for a single trained checkpoint, ready to
drop into the thesis. Change **one line** (the version cell) to document a
different version — every cell below follows.

**What it produces** (saved as PNG @300 dpi + PDF under `figures/<version>/`,
mirrored to Drive, and zipped for download):

| # | Figure | Source |
|---|--------|--------|
| — | Architecture diagram + text summary | `plot_model` |
| 01 | Clips-per-class (dataset composition) | mixer cache |
| 02 | Detection precision / recall / F1 per class | detection sweep |
| 03 | Detection totals + macro averages | detection sweep |
| 04 | Detection co-occurrence heatmap | detection sweep |
| 05 | ROC + Precision-Recall curves | detection scores |
| 06 | Score separability histogram | detection scores |
| 07 | Top false-positive classes | detection sweep |
| 08 | Threshold sweep (F1 / FP / TP vs cap) | detection scores |
| 09 | SI-SDRi per class | `evaluate_conditioned_separator` |
| 10 | SI-SDR mixture-vs-model | `evaluate_conditioned_separator` |
| 11 | FiLM class-discrimination per class | `diagnose_model` |
| 12 | Output/input magnitude ratio per class | `diagnose_model` |
| 13 | Qualitative spectrogram panels (mix / mask / est / target) | model |
| 14 | Removal demo (before/after waveform + spectrogram + audio) | `audio_engine` |
| — | `per_class_metrics.csv` / `.xlsx`, `summary.json` | merged |

It documents whatever `SNC_MODEL_VERSION` points at — works for any trained
checkpoint (v2.x or v3.0), with or without a detection head.

## 1. Mount Drive and clone the repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
# The evaluation/figure code is version-agnostic, so this branch can document
# any trained version. feature/v3.0 has the latest eval code.
BRANCH = 'feature/v3.0'

import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if token:
    clone_url = f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('Using GITHUB_TOKEN from Colab Secrets.')
else:
    clone_url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
    print('No GITHUB_TOKEN secret found - attempting anonymous clone.')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

result = subprocess.run(
    ['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
    capture_output=True, text=True, stdin=subprocess.DEVNULL,
    env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'},
)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'git clone failed (exit {result.returncode}).')
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'], check=True)

## 2. Pick the version to document

**This is the only line you change.** Set it to any trained checkpoint.

In [ ]:
# === CHANGE THIS ONE LINE ===========================================
os.environ['SNC_MODEL_VERSION'] = 'v3.0'   # e.g. 'v2.8', 'v2.6', 'v3.0'
# ====================================================================
os.environ['SNC_DATA_CACHE_DIR'] = '/content'

import sys
sys.path.insert(0, 'src/model_training')
import model_config as cfg

VERSION = cfg.model_version()
print('Documenting version :', VERSION)
print('Checkpoint          :', cfg.model_path())
print('Class names         :', cfg.classes_path())
print('Detection allow-list:', cfg.detect_allowlist_path())
print()
print('NOTE: the checkpoint must already be trained and present on Drive')
print('(saved_models is symlinked to Drive in the next cell).')

## 3. Symlink data + saved_models to Drive

In [ ]:
drive_data = Path(DRIVE_ROOT) / 'data'
drive_models = Path(DRIVE_ROOT) / 'saved_models'
drive_data.mkdir(parents=True, exist_ok=True)
drive_models.mkdir(parents=True, exist_ok=True)

local_models = REPO_DIR / 'saved_models'
if local_models.is_symlink() or local_models.exists():
    local_models.unlink() if local_models.is_symlink() else shutil.rmtree(local_models)
local_models.symlink_to(drive_models)
print(f'{local_models} -> {drive_models}')

raw_dir = REPO_DIR / 'data' / 'raw'
raw_dir.mkdir(parents=True, exist_ok=True)
for sub in ['archive', 'urbansound8k']:
    local = raw_dir / sub
    target = drive_data / 'raw' / sub
    target.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        shutil.rmtree(local)
    local.symlink_to(target)
    print(f'{local} -> {target}')

## 4. Datasets (download ESC-50 + UrbanSound8K if missing)

Figures use synthetic mixtures built from real ESC-50 / UrbanSound8K clips, so
the audio must be present. Already-downloaded datasets on Drive are reused.

In [ ]:
import zipfile, urllib.request

archive_dir = REPO_DIR / 'data' / 'raw' / 'archive'
csv_path = archive_dir / 'esc50.csv'
wav_dir = archive_dir / 'audio' / 'audio'
if csv_path.exists() and wav_dir.exists() and len(list(wav_dir.glob('*.wav'))) >= 2000:
    print(f'ESC-50 present - {len(list(wav_dir.glob("*.wav")))} WAVs.')
else:
    archive_dir.mkdir(parents=True, exist_ok=True)
    zip_path = Path('/content/esc50.zip')
    print('Downloading ESC-50...')
    urllib.request.urlretrieve(
        'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip', zip_path)
    extract_dir = Path('/content/esc50_extracted')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    src_root = extract_dir / 'ESC-50-master'
    shutil.copy(src_root / 'meta' / 'esc50.csv', csv_path)
    wav_dir.mkdir(parents=True, exist_ok=True)
    for wav in (src_root / 'audio').glob('*.wav'):
        shutil.copy(wav, wav_dir / wav.name)
    zip_path.unlink(); shutil.rmtree(extract_dir)
    print(f'ESC-50 done - {len(list(wav_dir.glob("*.wav")))} WAVs.')
assert csv_path.exists() and wav_dir.exists()

In [ ]:
import tarfile, time

us8k_dir = REPO_DIR / 'data' / 'raw' / 'urbansound8k'
us8k_csv = us8k_dir / 'metadata' / 'UrbanSound8K.csv'
if us8k_csv.exists():
    print('UrbanSound8K present.')
else:
    tar_path = Path('/content/urbansound8k.tar.gz')
    url = 'https://zenodo.org/record/1203745/files/UrbanSound8K.tar.gz'
    print('Downloading UrbanSound8K (~6 GB)...')
    have_wget = subprocess.run(['which', 'wget'], capture_output=True).returncode == 0
    if have_wget:
        subprocess.run(['wget', '--continue', '--tries=20', '--retry-connrefused',
                        '--waitretry=60', '--timeout=120', '--read-timeout=300',
                        '-O', str(tar_path), url], check=True)
    else:
        urllib.request.urlretrieve(url, tar_path)
    print('Extracting...')
    extract_dir = Path('/content/us8k_extract')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    with tarfile.open(tar_path) as t:
        t.extractall(extract_dir)
    us8k_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(str(extract_dir / 'UrbanSound8K'), str(us8k_dir))
    tar_path.unlink(); shutil.rmtree(extract_dir, ignore_errors=True)
    print('UrbanSound8K ready.')
assert us8k_csv.exists()

## 5. Install dependencies

In [ ]:
!pip -q install librosa==0.11.0 soundfile scikit-learn pandas openpyxl matplotlib pydot
# graphviz is needed by tf.keras.utils.plot_model for the architecture diagram.
!apt-get -qq install -y graphviz > /dev/null 2>&1 || echo 'graphviz apt install skipped'
import tensorflow as tf
print('TF', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Decoded-dataset cache: fetch from Drive or decode once, then mirror back.
os.environ['SNC_DATA_CACHE_DIR'] = '/content'
os.environ['SNC_DRIVE_CACHE_DIR'] = f'{DRIVE_ROOT}/cache'
!python scripts/prep_data_cache.py || echo 'cache prep skipped (will decode on first use)'

## 6. Setup — load model, classes, helpers, figure styling

All figures are saved through `save_fig`, which writes a 300-dpi PNG and a
vector PDF and records the path.

In [ ]:
%matplotlib inline
import sys, json, warnings
import tensorflow as tf
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
warnings.filterwarnings('ignore')

sys.path.insert(0, 'src/data_preparation')
sys.path.insert(0, 'src/model_training')
sys.path.insert(0, 'src/application/modern_web')
import model_config as cfg
from conditioned_separator import FREQ_BINS, TIME_FRAMES, HOP_LENGTH, N_FFT, SAMPLE_RATE
from separation_mixer import SeparationMixer, waveform_to_magnitude
import evaluate_detection as ed
import evaluate_conditioned_separator as ec
import diagnose_model as dm

VERSION = cfg.model_version()
MODEL_PATH = cfg.model_path()
assert MODEL_PATH.exists(), (
    f'Checkpoint not found: {MODEL_PATH}\n'
    f'Train this version first, or set SNC_MODEL_VERSION to a trained one.')
MODEL_CLASSES = json.load(cfg.classes_path().open())
ALLOWED = cfg.load_detect_allowlist()

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
HAS_DET = len(model.outputs) > 1
print(f'Version          : {VERSION}')
print(f'Parameters       : {model.count_params():,}')
print(f'Classes ({len(MODEL_CLASSES)})     : {MODEL_CLASSES}')
print(f'Detection head   : {HAS_DET}')
print(f'Allow-list       : {None if ALLOWED is None else len(ALLOWED)} classes')

FIG_DIR = Path('/content/thesis_figures') / VERSION
FIG_DIR.mkdir(parents=True, exist_ok=True)
GENERATED = []

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.grid': True,
    'grid.alpha': 0.3, 'axes.axisbelow': True, 'figure.autolayout': True,
})


def save_fig(fig, name, also_pdf=True):
    png = FIG_DIR / f'{name}.png'
    fig.savefig(png)
    GENERATED.append(png)
    if also_pdf:
        pdf = FIG_DIR / f'{name}.pdf'
        fig.savefig(pdf)
        GENERATED.append(pdf)
    plt.show()
    print('saved:', png.name, '(+pdf)' if also_pdf else '')


SEED = 7777
mixer = SeparationMixer(cfg.DATA_ROOT, negative_prob=0.0, seed=SEED)
AVAILABLE = [c for c in mixer.class_names if c in set(MODEL_CLASSES)]
print(f'Local audio for {len(AVAILABLE)}/{len(MODEL_CLASSES)} model classes.')


def one_window(c):
    # Deterministic 1-second window from the middle clip of class c.
    clip = mixer._waveform_cache[c][len(mixer._waveform_cache[c]) // 2]
    return mixer._random_window(clip)

## Architecture diagram + summary

In [ ]:
from contextlib import redirect_stdout
from IPython.display import Image, display

summ_path = FIG_DIR / 'model_summary.txt'
with summ_path.open('w') as f, redirect_stdout(f):
    model.summary(expand_nested=True, line_length=110)
GENERATED.append(summ_path)
print(f'Parameters: {model.count_params():,} | inputs: {len(model.inputs)} | outputs: {len(model.outputs)}')

try:
    arch_png = FIG_DIR / 'architecture.png'
    tf.keras.utils.plot_model(model, to_file=str(arch_png), show_shapes=True,
                              show_layer_names=True, dpi=120, expand_nested=True)
    GENERATED.append(arch_png)
    display(Image(str(arch_png)))
    print('saved:', arch_png.name)
except Exception as e:
    print(f'plot_model unavailable ({e}); text summary saved to {summ_path.name}.')

## 01 — Dataset composition (this version's vocabulary)

In [ ]:
counts = {c: len(mixer._waveform_cache[c]) for c in AVAILABLE}
order = sorted(counts, key=counts.get, reverse=True)
fig, ax = plt.subplots(figsize=(9, max(3, 0.33 * len(order))))
ax.barh(order[::-1], [counts[c] for c in order[::-1]], color='#4C72B0')
ax.set_xlabel('number of clips')
ax.set_title(f'Clips per class ({len(order)} classes)')
save_fig(fig, '01_dataset_clip_counts')

pd.DataFrame({'class': order, 'clips': [counts[c] for c in order]}
             ).to_csv(FIG_DIR / 'class_clip_counts.csv', index=False)
GENERATED.append(FIG_DIR / 'class_clip_counts.csv')

## Detection — one scoring pass

Builds `N_MIX` synthetic multi-class mixtures, scores every class with the
exact webapp logic (`detect_classes`), and stores both the decisions and the
raw scores. All detection figures (02-08) come from this one pass.

In [ ]:
N_MIX = 300
FLOOR, CAP, TOPK = ed.ABSOLUTE_FLOOR, ed.RELATIVE_CAP, ed.TOP_K

det_mixer = SeparationMixer(cfg.DATA_ROOT, negative_prob=0.0, seed=SEED)
avail = [c for c in det_mixer.class_names if c in set(MODEL_CLASSES)]
rng = det_mixer._rng
idx = {c: i for i, c in enumerate(avail)}

tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int); n_present = defaultdict(int)
cooc = np.zeros((len(avail), len(avail)), dtype=float)
y_true, y_score = [], []
RECORDS = []

for i in range(N_MIX):
    k = rng.randint(1, min(det_mixer.max_mix, len(avail)))
    present = set(rng.sample(avail, k))
    mix = np.zeros(det_mixer.window_length, dtype=np.float32)
    for c in present:
        clip = rng.choice(det_mixer._waveform_cache[c])
        mix += det_mixer._random_window(clip) * rng.uniform(*det_mixer.amp_range)
    mix = det_mixer._maybe_add_background_noise(mix)

    detected, scores = ed.detect_classes(model, MODEL_CLASSES, mix, ALLOWED, FLOOR, CAP, TOPK)
    RECORDS.append((present, {c: float(scores[c]) for c in avail}))

    for c in present:
        n_present[c] += 1
        (tp if c in detected else fn)[c] += 1
    for c in detected:
        if c not in present:
            fp[c] += 1
    for p in present:
        for d in detected:
            if d in idx:
                cooc[idx[p], idx[d]] += 1
    for c in avail:
        y_true.append(1 if c in present else 0)
        y_score.append(float(scores[c]))
    if (i + 1) % 50 == 0:
        print(f'  {i + 1}/{N_MIX} mixtures')

y_true, y_score = np.array(y_true), np.array(y_score)
rows = []
for c in sorted(avail):
    if n_present[c] == 0:
        continue
    prec = tp[c] / (tp[c] + fp[c]) if (tp[c] + fp[c]) else 0.0
    rec = tp[c] / (tp[c] + fn[c]) if (tp[c] + fn[c]) else 0.0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    rows.append(dict(cls=c, n=n_present[c], precision=prec, recall=rec, f1=f1,
                     tp=tp[c], fp=fp[c], fn=fn[c]))
det_df = pd.DataFrame(rows).sort_values('f1', ascending=False).reset_index(drop=True)
MACRO_F1 = float(det_df['f1'].mean()) if len(det_df) else 0.0
print(f'\nMacro F1 = {MACRO_F1:.3f} | TP={sum(tp.values())} '
      f'FP={sum(fp.values())} FN={sum(fn.values())} | detection head={HAS_DET}')
det_df

## 02 — Detection precision / recall / F1 per class

In [ ]:
d = det_df.sort_values('f1')
yp = np.arange(len(d))
fig, ax = plt.subplots(figsize=(9, max(3, 0.36 * len(d))))
ax.barh(yp - 0.27, d['precision'], height=0.27, label='precision', color='#55A868')
ax.barh(yp,        d['recall'],    height=0.27, label='recall',    color='#DD8452')
ax.barh(yp + 0.27, d['f1'],        height=0.27, label='F1',        color='#4C72B0')
ax.set_yticks(yp); ax.set_yticklabels(d['cls']); ax.set_xlim(0, 1)
ax.axvline(MACRO_F1, ls='--', c='k', alpha=0.6, label=f'macro F1={MACRO_F1:.2f}')
ax.set_title(f'Detection precision / recall / F1 per class')
ax.legend(loc='lower right')
save_fig(fig, '02_detection_per_class_prf')

## 03 — Detection totals + macro averages

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(['TP', 'FP', 'FN'],
            [sum(tp.values()), sum(fp.values()), sum(fn.values())],
            color=['#55A868', '#C44E52', '#CCB974'])
axes[0].set_title('Detection totals'); axes[0].set_ylabel('count')
macro = dict(precision=float(det_df['precision'].mean()),
             recall=float(det_df['recall'].mean()), f1=MACRO_F1)
axes[1].bar(list(macro), list(macro.values()), color='#4C72B0')
axes[1].set_ylim(0, 1); axes[1].set_title('Macro averages')
for i, v in enumerate(macro.values()):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center')
fig.suptitle(f'Detection summary')
save_fig(fig, '03_detection_totals')

## 04 — Detection co-occurrence heatmap

Row-normalised: cell *(present i, surfaced j)* = how often class **j** is
surfaced when class **i** is in the mixture. The diagonal approximates recall;
bright off-diagonal cells are systematic confusions.

In [ ]:
denom = np.maximum(np.array([n_present[c] for c in avail])[:, None], 1)
norm = cooc / denom
fig, ax = plt.subplots(figsize=(1.2 + 0.42 * len(avail), 1.2 + 0.42 * len(avail)))
im = ax.imshow(norm, cmap='viridis', vmin=0, vmax=1)
ax.set_xticks(range(len(avail))); ax.set_xticklabels(avail, rotation=90)
ax.set_yticks(range(len(avail))); ax.set_yticklabels(avail)
ax.set_xlabel('surfaced class'); ax.set_ylabel('present class')
ax.set_title(f'Detection co-occurrence (row-normalised)')
fig.colorbar(im, fraction=0.046, pad=0.04)
save_fig(fig, '04_detection_cooccurrence')

## 05 — ROC + Precision-Recall curves

Micro-averaged over every (mixture, class) presence decision, using the raw
detection score (presence probability for a detection-head model, energy
heuristic otherwise).

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

if 0 < y_true.sum() < len(y_true):
    fpr, tpr, _ = roc_curve(y_true, y_score); roc_auc = auc(fpr, tpr)
    prc, rec, _ = precision_recall_curve(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
    axes[0].plot(fpr, tpr, c='#4C72B0', lw=2, label=f'AUC = {roc_auc:.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
    axes[0].set_xlabel('false-positive rate'); axes[0].set_ylabel('true-positive rate')
    axes[0].set_title('ROC (micro)'); axes[0].legend(loc='lower right')
    axes[1].plot(rec, prc, c='#C44E52', lw=2, label=f'AP = {ap:.3f}')
    axes[1].set_xlabel('recall'); axes[1].set_ylabel('precision')
    axes[1].set_title('Precision-Recall (micro)'); axes[1].legend(loc='upper right')
    fig.suptitle(f'Detection ranking quality')
    save_fig(fig, '05_detection_roc_pr')
else:
    print('ROC/PR skipped (degenerate labels).')

## 06 — Score separability (present vs absent)

In [ ]:
pos = y_score[y_true == 1]; neg = y_score[y_true == 0]
lo, hi = float(min(y_score.min(), 0.0)), float(max(y_score.max(), 1.0))
bins = np.linspace(lo, hi, 40)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(neg, bins=bins, alpha=0.6, density=True, color='#C44E52', label=f'absent (n={len(neg)})')
ax.hist(pos, bins=bins, alpha=0.6, density=True, color='#55A868', label=f'present (n={len(pos)})')
ax.set_xlabel('detection score ' + ('(presence probability)' if HAS_DET else '(energy heuristic)'))
ax.set_ylabel('density'); ax.set_title(f'Score separability'); ax.legend()
save_fig(fig, '06_detection_score_hist')

## 07 — Top false-positive classes

In [ ]:
top_fp = [(c, n) for c, n in sorted(fp.items(), key=lambda kv: kv[1], reverse=True)[:12] if n > 0]
if top_fp:
    fig, ax = plt.subplots(figsize=(9, max(3, 0.42 * len(top_fp))))
    ax.barh([c for c, _ in top_fp][::-1], [n for _, n in top_fp][::-1], color='#C44E52')
    ax.set_xlabel('false positives'); ax.set_title(f'Top false-positive classes')
    save_fig(fig, '07_detection_top_fp')
else:
    print('No false positives to plot.')

## 08 — Detection threshold sweep

Recomputed from the stored scores (no model re-runs): sweeps `relative_cap` and
shows macro F1, total FP and total TP. The dotted line marks the default cap.

In [ ]:
def decide(scores, floor, cap, topk):
    ranked = sorted(scores, key=scores.get, reverse=True)
    if not ranked:
        return set()
    cutoff = max(floor, cap * scores[ranked[0]])
    return set([n for n in ranked if scores[n] >= cutoff][:topk])

caps = np.round(np.arange(0.50, 0.99, 0.05), 2)
mf1, tot_fp, tot_tp = [], [], []
for cap in caps:
    t = defaultdict(int); f = defaultdict(int); fnn = defaultdict(int); npr = defaultdict(int)
    for present, scores in RECORDS:
        det = decide(scores, FLOOR, cap, TOPK)
        for c in present:
            npr[c] += 1
            (t if c in det else fnn)[c] += 1
        for c in det:
            if c not in present:
                f[c] += 1
    f1s = []
    for c in npr:
        pr = t[c] / (t[c] + f[c]) if (t[c] + f[c]) else 0.0
        rc = t[c] / (t[c] + fnn[c]) if (t[c] + fnn[c]) else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if (pr + rc) else 0.0)
    mf1.append(float(np.mean(f1s))); tot_fp.append(sum(f.values())); tot_tp.append(sum(t.values()))

fig, ax1 = plt.subplots(figsize=(9, 4.6))
ax1.plot(caps, mf1, 'o-', c='#4C72B0', label='macro F1')
ax1.set_xlabel('relative_cap'); ax1.set_ylabel('macro F1', color='#4C72B0'); ax1.set_ylim(0, 1)
ax2 = ax1.twinx(); ax2.grid(False)
ax2.plot(caps, tot_fp, 's--', c='#C44E52', label='total FP')
ax2.plot(caps, tot_tp, '^--', c='#55A868', label='total TP')
ax2.set_ylabel('count')
ax1.axvline(CAP, ls=':', c='k', alpha=0.6)
ax1.set_title(f'Threshold sweep (top_k={TOPK}, floor={FLOOR})')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='center right')
save_fig(fig, '08_detection_threshold_sweep')

## 09 / 10 — Separation quality (SI-SDR)

Runs the SI-SDR evaluator on positive-only mixtures. **SI-SDRi** (model minus
mixture baseline, dB) is the headline separation metric.

In [ ]:
sisdr = ec.ConditionedSeparatorEvaluator(cfg.DATA_ROOT, MODEL_PATH, n_test=400, seed=4242).evaluate()
si_df = pd.DataFrame(sisdr['per_class']).sort_values('si_sdri', ascending=False).reset_index(drop=True)

s = si_df.sort_values('si_sdri')
fig, ax = plt.subplots(figsize=(9, max(3, 0.36 * len(s))))
ax.barh(s['class'], s['si_sdri'],
        color=['#55A868' if v >= 0 else '#C44E52' for v in s['si_sdri']])
ax.axvline(sisdr['si_sdri'], ls='--', c='k', alpha=0.6, label=f"mean = {sisdr['si_sdri']:.2f} dB")
ax.set_xlabel('SI-SDRi (dB)'); ax.set_title(f'Separation SI-SDRi per class'); ax.legend()
save_fig(fig, '09_sisdr_per_class')

In [ ]:
s2 = si_df.sort_values('si_sdr_model')
yp = np.arange(len(s2))
fig, ax = plt.subplots(figsize=(9, max(3, 0.36 * len(s2))))
ax.barh(yp - 0.2, s2['si_sdr_mix'], height=0.4, label='mixture (baseline)', color='#CCB974')
ax.barh(yp + 0.2, s2['si_sdr_model'], height=0.4, label='model', color='#4C72B0')
ax.set_yticks(yp); ax.set_yticklabels(s2['class']); ax.set_xlabel('SI-SDR (dB)')
ax.set_title(f'SI-SDR: mixture vs model'); ax.legend()
save_fig(fig, '10_sisdr_mix_vs_model')

## 11 / 12 — Model diagnostics (FiLM conditioning)

Per class, compares the output energy for the **correct** query against
**wrong** queries (does the FiLM conditioning isolate the queried class?), and
the output/input magnitude ratio (is the model producing non-trivial output?).
Advantage is shown on a signed-log scale because it can span several orders of
magnitude.

In [ ]:
adv_rows = []
for c in AVAILABLE:
    clip = dm._clean_clip(mixer, c)
    lin, log = dm._model_input(clip)
    est_c = dm._predict_mask(model, log, dm._query(MODEL_CLASSES, c), lin)
    correct_e = float(np.mean(est_c ** 2))
    out_in = float(np.max(est_c)) / (float(np.max(lin)) + 1e-8)
    wrongs = [w for w in AVAILABLE if w != c][:3]
    we = [float(np.mean(dm._predict_mask(model, log, dm._query(MODEL_CLASSES, w), lin) ** 2))
          for w in wrongs]
    wrong_e = float(np.mean(we)) if we else 0.0
    adv = (correct_e - wrong_e) / (wrong_e + 1e-9)
    adv_rows.append(dict(cls=c, advantage=adv, out_in=out_in,
                         correct_energy=correct_e, wrong_energy=wrong_e))
diag_df = pd.DataFrame(adv_rows)

g = diag_df.sort_values('advantage')
disp = np.sign(g['advantage']) * np.log10(1.0 + np.abs(g['advantage']))
fig, ax = plt.subplots(figsize=(9, max(3, 0.36 * len(g))))
ax.barh(g['cls'], disp, color=['#55A868' if v > 0.3 else '#C44E52' for v in g['advantage']])
ax.axvline(0, c='k', lw=0.8)
ax.set_xlabel('signed log10(1 + |advantage|)')
ax.set_title(f'FiLM class discrimination (correct vs wrong query)')
save_fig(fig, '11_film_advantage')

In [ ]:
g2 = diag_df.sort_values('out_in')
fig, ax = plt.subplots(figsize=(9, max(3, 0.36 * len(g2))))
ax.barh(g2['cls'], g2['out_in'], color='#4C72B0')
ax.axvline(0.10, ls='--', c='k', alpha=0.5, label='health floor 0.10')
ax.set_xlabel('output / input magnitude ratio')
ax.set_title(f'Output magnitude per class'); ax.legend()
save_fig(fig, '12_out_in_ratio')

## 13 — Qualitative spectrogram panels

For a few well-detected classes: the input mixture, the predicted mask, the
estimated stem, and the ground-truth stem.

In [ ]:
def spec_panel(target, distractors):
    tw = one_window(target).copy()
    mix = tw.copy()
    for dist in distractors:
        mix = mix + one_window(dist) * 0.7
    peak = float(np.max(np.abs(mix)))
    if peak > 1e-6:
        mix = mix / peak; tw = tw / peak
    lin = waveform_to_magnitude(mix)[..., None]
    log = np.log1p(lin)
    q = np.zeros(len(MODEL_CLASSES), np.float32); q[MODEL_CLASSES.index(target)] = 1.0
    pred = model.predict([log[None], q[None], lin[None]], verbose=0)
    est = (pred[0] if isinstance(pred, list) else pred)[0, :, :, 0]
    mask = np.clip(est / (lin[:, :, 0] + 1e-8), 0, 1)
    tgt = waveform_to_magnitude(tw)
    lg = lambda x: np.log1p(x)
    panels = [(lg(lin[:, :, 0]), 'mixture (log-mag)', 'magma'),
              (mask, 'predicted mask', 'viridis'),
              (lg(est), 'estimated stem', 'magma'),
              (lg(tgt), 'ground-truth stem', 'magma')]
    fig, axs = plt.subplots(1, 4, figsize=(16, 3.6))
    for a, (img, ttl, cmap) in zip(axs, panels):
        a.imshow(img, origin='lower', aspect='auto', cmap=cmap)
        a.set_title(ttl); a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'{target}  (mixed with {", ".join(distractors)})')
    save_fig(fig, f'13_spectrogram_{target}')

targets = list(det_df['cls'][:3]) if len(det_df) else AVAILABLE[:3]
for tgt_cls in targets:
    distr = [c for c in AVAILABLE if c != tgt_cls][:2]
    try:
        spec_panel(tgt_cls, distr)
    except Exception as e:
        print(f'panel {tgt_cls} failed: {e}')

## 14 — Removal demo (real pipeline: before / after)

Synthesises a multi-second mixture, runs the **actual** webapp removal engine
(`audio_engine.process`), and shows the before/after waveforms and spectrograms
plus inline audio players.

In [ ]:
import soundfile as sf
import audio_engine as ax_engine
from IPython.display import Audio

MODEL_STEM = cfg.model_stem()


def long_mix(c, secs=4):
    return np.concatenate([one_window(c) for _ in range(secs)])

target = targets[0] if targets else AVAILABLE[0]
distr = next(c for c in AVAILABLE if c != target)
mix = long_mix(target) + 0.7 * long_mix(distr)
mix = mix / (np.max(np.abs(mix)) + 1e-8)
demo_wav = FIG_DIR / f'demo_mixture_{target}.wav'
sf.write(str(demo_wav), mix, SAMPLE_RATE); GENERATED.append(demo_wav)

try:
    orig, cleaned, stems = ax_engine.process(
        str(demo_wav), MODEL_STEM, [{'name': target, 'strength': 1.0}])
    cw = FIG_DIR / f'demo_cleaned_{target}.wav'
    sf.write(str(cw), cleaned, SAMPLE_RATE); GENERATED.append(cw)

    fig, axs = plt.subplots(2, 2, figsize=(13, 6))
    axs[0, 0].plot(np.arange(len(orig)) / SAMPLE_RATE, orig, lw=0.5, c='#4C72B0')
    axs[0, 0].set_title('original waveform'); axs[0, 0].set_xlabel('s')
    axs[0, 1].plot(np.arange(len(cleaned)) / SAMPLE_RATE, cleaned, lw=0.5, c='#55A868')
    axs[0, 1].set_title(f'cleaned ({target} removed)'); axs[0, 1].set_xlabel('s')
    for a, (w, ttl) in zip([axs[1, 0], axs[1, 1]], [(orig, 'original'), (cleaned, 'cleaned')]):
        S = np.log1p(np.abs(librosa.stft(w, n_fft=N_FFT, hop_length=HOP_LENGTH))[:FREQ_BINS])
        a.imshow(S, origin='lower', aspect='auto', cmap='magma')
        a.set_title(f'{ttl} spectrogram'); a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f'Removal demo: {target} (mixed with {distr})')
    save_fig(fig, '14_removal_demo')

    print('Original:'); display(Audio(orig, rate=SAMPLE_RATE))
    print(f'Cleaned ({target} removed):'); display(Audio(cleaned, rate=SAMPLE_RATE))
    if stems:
        print(f'Extracted stem ({target}):'); display(Audio(stems[0][1], rate=SAMPLE_RATE))
except Exception as e:
    print(f'Removal demo failed ({e}).')

## Export — tables, summary, zip, download

Merges the per-class metrics, writes CSV / Excel / `summary.json`, mirrors
everything to Drive, and downloads a zip of `figures/<version>/`.

In [ ]:
merged = (det_df.rename(columns={'cls': 'class'})
          .merge(si_df, on='class', how='outer')
          .merge(diag_df.rename(columns={'cls': 'class'}), on='class', how='outer'))
merged_path = FIG_DIR / 'per_class_metrics.csv'
merged.to_csv(merged_path, index=False); GENERATED.append(merged_path)

try:
    xlsx = FIG_DIR / 'per_class_metrics.xlsx'
    with pd.ExcelWriter(xlsx) as xw:
        det_df.to_excel(xw, sheet_name='detection', index=False)
        si_df.to_excel(xw, sheet_name='si_sdr', index=False)
        diag_df.to_excel(xw, sheet_name='diagnostics', index=False)
        merged.to_excel(xw, sheet_name='merged', index=False)
    GENERATED.append(xlsx)
except Exception as e:
    print(f'xlsx export skipped ({e})')

summary = dict(version=VERSION, num_classes=len(MODEL_CLASSES),
               has_detection_head=bool(HAS_DET), macro_f1=MACRO_F1,
               tp=int(sum(tp.values())), fp=int(sum(fp.values())),
               fn=int(sum(fn.values())), si_sdri=float(sisdr['si_sdri']))
(FIG_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
GENERATED.append(FIG_DIR / 'summary.json')
print(json.dumps(summary, indent=2))

drive_out = Path(DRIVE_ROOT) / 'figures' / VERSION
drive_out.mkdir(parents=True, exist_ok=True)
for p in FIG_DIR.iterdir():
    shutil.copy(p, drive_out / p.name)
print(f'\nMirrored {len(list(FIG_DIR.iterdir()))} files to {drive_out}')

zip_base = f'/content/thesis_figures_{VERSION}'
shutil.make_archive(zip_base, 'zip', FIG_DIR)
print(f'Zip: {zip_base}.zip')
try:
    from google.colab import files
    files.download(f'{zip_base}.zip')
except Exception as e:
    print(f'(auto-download unavailable: {e}; grab it from Drive: {drive_out})')

print('\nGenerated files:')
for p in sorted(set(str(x) for x in GENERATED)):
    print('  ', p)